# Raw ERSP Clustering
210_raw_clustering.ipynb


K-Means + Hierarchical (Ward) clustering on raw flattened ERSP matrices (300×128 → 38400 features).

Outputs land in `outputs/clustering/{kmeans,hierarchical}/raw/runs/<timestamp>/` via `lf_cluster_run.fit_and_save`.
Each run writes: model.joblib, predictor.joblib, X_train.npy, labels.csv, metrics.json, manifest.json, plus standard figures (centroids, silhouette per cluster, similarity heatmap, centroid distance heatmap).


In [12]:
## 0. Config & imports
import os
from pathlib import Path
import datetime
import json

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import matplotlib.pyplot as plt  # for optional plots (no seaborn)

# ---- Identity / tags ----
SCRIPT_NAME = "210_raw_clustering.ipynb"
ALGO_TAG = "kmeans_raw"
RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")  # e.g. 20251117_112233

# ---- Paths ----
ANALYSIS_ROOT = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda")

# Input: RAWONLY ERSP matrices
INPUT_DIR = ANALYSIS_ROOT / r"01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY"

# Output: clustering results & figures
FIGURES_EMBEDDINGS_DIR = ANALYSIS_ROOT / r"02_FBM_Clustering/outputs/figures/embeddings"
FIGURES_ERSP_DIR       = ANALYSIS_ROOT / r"02_FBM_Clustering/outputs/figures/ersp_clusters"
LOG_DIR                = ANALYSIS_ROOT / r"02_FBM_Clustering/outputs/logs"

for d in [FIGURES_EMBEDDINGS_DIR, FIGURES_ERSP_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)


# WM electrodes for double check    
# ---- WM reference maps (unchanged data, compact helpers) ----



# ---- Data shape constants ----
N_TIME = 300
N_FREQ = 129
N_FEATURES = N_TIME * N_FREQ

# ---- Clustering parameters ----
N_CLUSTERS = 13
RANDOM_STATE = 42

print("SCRIPT_NAME:", SCRIPT_NAME)
print("ALGO_TAG   :", ALGO_TAG)
print("RUN_ID     :", RUN_ID)
print("Input dir  :", INPUT_DIR)


## Auto-detect patient IDs from outputs/04_ersp_LM_RAWONLY

from pathlib import Path

# Patient folders = all directories inside INPUT_DIR
PATIENT_IDS = sorted([d.name for d in INPUT_DIR.iterdir() if d.is_dir()])

print("Detected patient folders:")
for p in PATIENT_IDS:
    print("  -", p)

print("\nTotal patients detected:", len(PATIENT_IDS))


SCRIPT_NAME: 210_raw_clustering.ipynb
ALGO_TAG   : kmeans_raw
RUN_ID     : 20260602_105010
Input dir  : \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\outputs\04_ersp_LM_RAWONLY
Detected patient folders:
  - EL030
  - EL033
  - EL034
  - EL035
  - EL036
  - EL037
  - EL038
  - EL040
  - EL042
  - EL043
  - EL044
  - EL045
  - PAT_2868
  - PAT_3066
  - PAT_3301
  - PAT_3390
  - PAT_3415
  - PAT_3455
  - PAT_3780
  - PAT_3965
  - PAT_3975
  - PAT_5515
  - PAT_5533
  - PAT_6684
  - PAT_6704
  - PAT_6854

Total patients detected: 26


In [13]:
## 2. Helper: parse electrode name from filename

def parse_electrode_from_filename(fname: str) -> str:
    """
    Example filename:
        PAT_3301_picture_None_ERSP_AG2_TN.npy
    We want to extract 'AG2' as electrode name.
    """
    name = Path(fname).name
    # Split on '_ERSP_' first
    if "_ERSP_" in name:
        left, right = name.split("_ERSP_", 1)
        # right is like "AG2_TN.npy"
        # remove suffix and trailing parts after electrode
        right_no_ext = right.rsplit(".", 1)[0]  # remove .npy
        # Typically pattern is "<electrode>_TN" or similar
        electrode = right_no_ext.split("_")[0]
        return electrode
    else:
        # fallback: no ERSP marker, just strip extension
        return name.rsplit(".", 1)[0]


## Load canonical dataset (shared across 210/230/231/232)

In [14]:
# ── Canonical dataset (shared by 210/230/231/232) ──────────────────
# Loads ERSPs from INPUT_DIR, drops non-neural channels, then gates by
# high-activity. SAME filter for every clustering notebook so cross-
# feature-set / cross-method comparisons are on the IDENTICAL sample set.
# Cached in 02_FBM_Clustering/outputs/_dataset/canonical/ — subsequent
# notebook runs load instantly instead of re-walking the ERSP_matrix tree.
from functions.lf_dataset import prepare_dataset, DEFAULT_CACHE_DIR

INPUT_DIR = Path(r'\\\\nasac-m2.unige.ch\\m-HumanNeuronLab\\ANALYSIS\\FLM\\Analysis_LoraFanda\\01_FBM_Analysis\\outputs\\04_ersp_LM_RAWONLY')
if not INPUT_DIR.exists():
    INPUT_DIR = Path('../01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY').resolve()

df_meta, ersp_list, X_3d = prepare_dataset(INPUT_DIR, cache_dir=DEFAULT_CACHE_DIR)
print(f'\nCanonical dataset: {len(df_meta)} samples · X_3d.shape={X_3d.shape}')


[lf_dataset cache hit] \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\_dataset\canonical
  1538 samples · X_3d.shape=(1538, 129, 300)

Canonical dataset: 1538 samples · X_3d.shape=(1538, 129, 300)


In [15]:
## 4. Build X_raw (flattened) and standardize

if len(ersp_list) == 0:
    raise RuntimeError("No ERSP data loaded. Check paths and file patterns.")

# Stack into (n_samples, N_TIME, N_FREQ)
# X_3d already returned by prepare_dataset (n_samples, N_FREQ, N_TIME)
assert X_3d.ndim == 3, f'Unexpected X_3d shape {X_3d.shape}'  # shape: (n_samples, 300, 128)
print("X_3d shape:", X_3d.shape)

# Flatten each matrix into a single vector (row-major)
X_raw = X_3d.reshape(X_3d.shape[0], -1)  # shape: (n_samples, 38400)
print("X_raw shape:", X_raw.shape)

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
print("X_scaled shape:", X_scaled.shape)

# Quick sanity check on a subset of features
print("Mean (first 10 features):", X_scaled[:,:10].mean(axis=0))
print("Std  (first 10 features):", X_scaled[:, :10].std(axis=0))


X_3d shape: (1538, 129, 300)
X_raw shape: (1538, 38700)
X_scaled shape: (1538, 38700)
Mean (first 10 features): [ 2.8445911e-08  5.3275526e-09 -3.2786431e-08  6.7704363e-08
  7.0959757e-08 -4.9024624e-09  3.0209247e-08  2.7045898e-08
 -4.6428067e-08  2.7128251e-09]
Std  (first 10 features): [0.99999946 0.99999994 0.9999999  0.9999996  1.0000001  1.0000001
 1.         1.0000007  0.9999999  1.0000002 ]


# Clustering

Two methods on the same `X_scaled`: K-Means (with K sweep) and Hierarchical (Ward).
Each `fit_and_save` call writes a self-contained run directory and updates `outputs/clustering/index.json`.


In [ ]:
from functions import lf_cluster_run as R

# Sweep K to let silhouette pick the best, but cap to a reasonable range
K_RANGE = [10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

manifest_km = R.fit_and_save(
    X_scaled,
    df_keep=df_meta,
    method='kmeans',
    feature_set='raw',
    params={'k_range': K_RANGE, 'random_state': RANDOM_STATE, 'n_init': 20},
    scaler=scaler,
    method_label='K-Means',
    feature_set_label='Raw ERSP',
    notebook=SCRIPT_NAME,
)
BEST_K = manifest_km['summary']['best_k']
print(f"Best K (KMeans, by silhouette): {BEST_K}")


  K= 10  sil=0.0414


In [ ]:
# Hierarchical (Ward) — K-sweep so MOBA can scrub. linkage_Z is saved once and
# every K cut goes into cluster_labels_by_k.csv. Best K (by silhouette) is the
# default cut in labels.csv.
manifest_hc = R.fit_and_save(
    X_scaled,
    df_keep=df_meta,
    method='hierarchical',
    feature_set='raw',
    params={'linkage':'ward', 'metric': 'euclidean', 'k_range': K_RANGE},
    scaler=scaler,
    method_label='Hierarchical (Ward)',
    feature_set_label='Raw ERSP',
    notebook=SCRIPT_NAME,
)
print(f'Best K (HC/raw, by silhouette): {manifest_hc["summary"]["best_k"]}')


## ERSP-shaped centroid grids

The orchestrator's `centroids.png` is a generic cluster × feature heatmap. For raw ERSP,
the features have a natural (N_FREQ, N_TIME) shape, so we render a method-specific grid below
for both runs.


In [ ]:
# TODO: NEeds to be transformed 

from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

def _ersp_centroid_grid(manifest, X_3d, *, n_cols=5, vlim=5.0):
    """Reload labels for a saved run, compute per-cluster mean ERSP, plot grid."""
    run_dir = Path('\\\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\\02_FBM_Clustering\\outputs\\clustering') / manifest['method'] \
        / manifest['feature_set'] / 'runs' / manifest['run_id']
    labels = pd.read_csv(run_dir / 'labels.csv')['cluster_'+manifest['method']+'_'+manifest['feature_set']].to_numpy()
    uniq = np.unique(labels)
    k = len(uniq)
    n_rows = int(np.ceil(k / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3*n_cols, 2.5*n_rows))
    axes = np.atleast_1d(axes).reshape(-1)
    for i, c in enumerate(uniq):
        idx = np.where(labels == c)[0]
        # FIX: same bug as BACKFILL_CENTROIDS — X_3d is (n, N_FREQ, N_TIME) already,
        # so .reshape(N_TIME, N_FREQ) scrambles the buffer. Drop the reshape and the .T below.
        mean_ersp = X_3d[idx].mean(axis=0)
        ax = axes[i]
        ax.imshow(mean_ersp, aspect='auto', origin='lower',
                  cmap='bwr', vmin=-vlim, vmax=vlim, interpolation='nearest')
        ax.set_title(f'C{c}  n={len(idx)}', fontsize=8)
        ax.set_xlabel('time'); ax.set_ylabel('freq')
    for ax in axes[k:]:
        ax.axis('off')
    plt.suptitle(f"{manifest['method_label']} — cluster mean ERSPs", y=1.01, fontsize=10)
    plt.tight_layout()
    out_path = run_dir / 'cluster_mean_ersps.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'[saved] {out_path}')

_ersp_centroid_grid(manifest_km, X_3d)
_ersp_centroid_grid(manifest_hc, X_3d)


## Inspect cluster cohesion + separation

The orchestrator already saved these figures into each run dir; display them inline for quick comparison.
- **silhouette_per_cluster.png** — which clusters are tight; which are catch-alls.
- **similarity_heatmap.png** — pairwise distance reordered by cluster; tight block-diagonal = good.
- **centroid_distance_heatmap.png** — inter-cluster separation.


In [ ]:
# from IPython.display import Image, display, Markdown
# from pathlib import Path

# def _show_run_figures(manifest):
#     run_dir = Path('02_FBM_Clustering/outputs/clustering') / manifest['method'] \
#         / manifest['feature_set'] / 'runs' / manifest['run_id']
#     display(Markdown(f"### {manifest['method_label']} · {manifest['feature_set_label']} · run {manifest['run_id']}"))
#     s = manifest['summary']
#     display(Markdown(
#         f"- n_samples = **{s['n_samples']}**, n_clusters = **{s['n_clusters']}**\n"
#         f"- silhouette = **{s['silhouette_overall']:.3f}**, "
#         f"calinski_harabasz = **{s['calinski_harabasz']:.1f}**, "
#         f"davies_bouldin = **{s['davies_bouldin']:.3f}**\n"
#     ))
#     for name in ['silhouette_per_cluster.png', 'similarity_heatmap.png',
#                  'centroid_distance_heatmap.png']:
#         p = run_dir / name
#         if p.exists():
#             display(Image(filename=str(p)))

# _show_run_figures(manifest_km)
# _show_run_figures(manifest_hc)


## Per-cluster centroid PNGs (for the MOBA cluster chips)

Re-runs cheap: no fitting, just `mean(X_3d[labels==c]).reshape(...)` and
`imshow` per cluster. Iterates every run in `index.json` and writes
`<run_dir>/cluster_centroids/cluster_<NN>.png` (one per cluster) so the
MOBA cluster filter chips can show a tiny mean-ERSP thumbnail per cluster.
Skips any run whose `feature_set` isn't `raw` (the others have a different
feature shape that doesn't make sense to display as an ERSP).


In [ ]:
# BACKFILL_CENTROIDS — per-cluster mean-ERSP thumbnails for the MOBA chips.
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

CLUSTERING_DIR = Path('\\\\nasac-m2.unige.ch\\m-HumanNeuronLab\\ANALYSIS\\FLM\\Analysis_LoraFanda\\02_FBM_Clustering\\outputs\\clustering')
INDEX_PATH = CLUSTERING_DIR / 'index.json'


def _save_per_cluster_centroid_pngs(manifest, X_3d_local, *, vlim=5.0):
    """For one run, write `<run_dir>/cluster_centroids/cluster_<NN>.png`."""
    if manifest['feature_set'] != 'raw':
        return 0
    run_dir = CLUSTERING_DIR / manifest['method'] / manifest['feature_set'] \
        / 'runs' / manifest['run_id']
    cluster_col = f"cluster_{manifest['method']}_{manifest['feature_set']}"
    df = pd.read_csv(run_dir / 'labels.csv')
    if cluster_col not in df.columns:
        # Fall back to whichever cluster_* column exists
        cands = [c for c in df.columns if c.startswith('cluster_')]
        if not cands: return 0
        cluster_col = cands[0]

    labels = df[cluster_col].to_numpy()
    if len(labels) != X_3d_local.shape[0]:
        print(f"  [skip] {manifest['run_id']}: labels ({len(labels)}) vs X_3d "
              f"({X_3d_local.shape[0]}) length mismatch — re-run 210 data load "
              f"with the same patient set as this run")
        return 0

    out_dir = run_dir / 'cluster_centroids'
    out_dir.mkdir(parents=True, exist_ok=True)

    uniq = sorted(int(c) for c in np.unique(labels))
    for c in uniq:
        idx = np.where(labels == c)[0]
        # FIX: don't reshape — mean is already (N_FREQ, N_TIME). Old reshape(N_TIME, N_FREQ)
        # was scrambling the buffer and producing vertical-stripe garbage in the chip thumbnails.
        mean_ersp = X_3d_local[idx].mean(axis=0)
        fig, ax = plt.subplots(figsize=(2, 1.5))
        ax.imshow(mean_ersp, aspect='auto', origin='lower',
                  cmap='bwr', vmin=-vlim, vmax=vlim, interpolation='nearest')
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values(): s.set_visible(False)
        fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
        out_path = out_dir / f'cluster_{int(c):02d}.png'
        fig.savefig(out_path, dpi=80, bbox_inches='tight', pad_inches=0)
        plt.close(fig)
    return len(uniq)


# Iterate every run in the index and backfill
if not INDEX_PATH.exists():
    print(f'No {INDEX_PATH} yet — run 210 cell 11/12 first.')
else:
    with open(INDEX_PATH) as f:
        idx = json.load(f)
    runs = idx.get('runs', [])
    print(f'Backfilling per-cluster centroid PNGs for {len(runs)} runs...')
    for run in runs:
        run_dir = CLUSTERING_DIR / run['path']
        manifest_path = run_dir / 'manifest.json'
        if not manifest_path.exists():
            print(f"  [skip] {run['path']}: missing manifest.json"); continue
        with open(manifest_path) as f:
            manifest = json.load(f)
        n = _save_per_cluster_centroid_pngs(manifest, X_3d)
        if n:
            print(f"  [{manifest['method']}/{manifest['feature_set']}] "
                  f"{manifest['run_id']}  ->  {n} cluster PNGs")
    print('\nDone. Commit and push:')
    print('  git add 02_FBM_Clustering/outputs/clustering')
    print('  git commit -m "Per-cluster centroid PNGs for MOBA chips"')
    print('  git push')
